# Risk rating via `lic_dsf.rating`

CI Summary thresholds, Chart Data mechanical ratings, Output 7.
See `docs/10-risk-rating.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.dsa import BaselineExternalBook, BaselinePublicBook
from lic_dsf.pv import (
    ExternalDebtBook,
    MacroDebtBook,
    PVPortfolio,
    load_external_debt_inputs,
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
    load_macro_debt_inputs,
)
from lic_dsf.rating import (
    ChartDataRegistry,
    RiskRatingSummary,
    compute_mechanical_ratings,
    load_ci_summary,
    risk_summary_panel,
)

for cand in (Path.cwd(), Path.cwd().parent):
    if (cand / "data" / "lic-dsf-template-2025-08-12.xlsx").exists():
        REPO = cand
        break
WB = REPO / "data" / "lic-dsf-template-2025-08-12.xlsx"

snap = load_ci_summary(WB)
print(snap.country, snap.dcc, snap.ci_score)
print(snap.thresholds)

instruments = load_instruments_from_workbook(WB, include_zero_disbursement=True)
lc_nr = load_lc_nr_instruments_from_workbook(WB, include_zero_disbursement=True)
ext = ExternalDebtBook(
    portfolio=PVPortfolio(instruments=tuple(instruments) + tuple(lc_nr)),
    inputs=load_external_debt_inputs(WB),
)
macro = MacroDebtBook(inputs=load_macro_debt_inputs(WB), external=ext)
ext_b = BaselineExternalBook(macro=macro, external=ext)
pub_b = BaselinePublicBook(macro=macro, external=ext)

years = [y for y in ext_b.years if y >= macro.inputs.first_projection_year][:11]
registry = ChartDataRegistry()
registry.register_series(
    "pv_debt_to_gdp", "baseline", ext_b.pv_ppg_external_to_gdp().reindex(years), is_baseline=True
)
registry.register_series(
    "pv_debt_to_exports", "baseline", ext_b.pv_ppg_external_to_exports().reindex(years), is_baseline=True
)
registry.register_series(
    "debt_service_to_exports", "baseline", ext_b.ppg_debt_service_to_exports().reindex(years), is_baseline=True
)
registry.register_series(
    "debt_service_to_revenue", "baseline", ext_b.ppg_debt_service_to_revenue().reindex(years), is_baseline=True
)
registry.register_series(
    "public_pv_debt_to_gdp", "baseline", pub_b.pv_public_debt_to_gdp().reindex(years), is_baseline=True
)

mech = compute_mechanical_ratings(registry, snap.thresholds, years=years)
summary = RiskRatingSummary(
    mechanical=mech, thresholds=snap.thresholds, dcc=snap.dcc, ci_score=snap.ci_score
)
risk_summary_panel(summary)

